# Mixture of Experts - Data Preparation
## Objective: Prepare telemetry data for MoE anomaly detection system

This notebook prepares the raw telemetry data (164 variables) for training 4 specialized GMM experts:
- **Expert 1**: Tire Dynamics (20 features)
- **Expert 2**: Vehicle Dynamics (16 features)
- **Expert 3**: Driver Control (6 features)
- **Expert 4**: Power Systems (4 features)
- **Total**: 46 features

**Note**: Variables with zero variance have been removed:
- **Expert 3**: TC_InAction, ABS_InAction, BrakeBias, BrakeTemp_*
- **Expert 4**: Fuel, EngineTemp_Oil, CurrentMaxRpm, EngineBrake, ERS_PowerLevel, ERS_RecoveryLevel

**Input**: `data/raw/telemetry_2025-12-08_18-05-21.csv`

**Output**: 
- `data/processed/MoE-anomaly/splits/expert{1-4}_{train,val,test}.csv`
- `data/processed/MoE-anomaly/scalers/scaler_expert{1-4}.pkl`
- `data/processed/MoE-anomaly/metadata/feature_stats.json`

---

## Section 0: Imports and Configuration

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import json
import pickle

# Set random seed for reproducibility
SEED = 42
np.random.seed(SEED)

In [2]:
# Configuration
ROOT = Path.cwd().parents[1]
DATA_DIR = ROOT / 'data'
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed' / 'MoE-anomaly'

# Input file
INPUT_FILE = RAW_DIR / 'telemetry_2025-12-08_18-05-21.csv'

# Create output directories
SPLITS_DIR = PROCESSED_DIR / 'splits'
SCALERS_DIR = PROCESSED_DIR / 'scalers'
FIGS_DIR = PROCESSED_DIR / 'figs'
METADATA_DIR = PROCESSED_DIR / 'metadata'

for dir_path in [SPLITS_DIR, SCALERS_DIR, FIGS_DIR, METADATA_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

# Split ratios
SPLIT_RATIOS = (0.70, 0.15, 0.15)  # train, val, test

print(f"Input file: {INPUT_FILE}")
print(f"Output directory: {PROCESSED_DIR}")
print(f"Split ratios: {SPLIT_RATIOS}")

Input file: c:\Users\victo\Desktop\Documents\Cuarto Año\Primer Cuatrimestre\F1_AC_Digital_Twin\data\raw\telemetry_2025-12-08_18-05-21.csv
Output directory: c:\Users\victo\Desktop\Documents\Cuarto Año\Primer Cuatrimestre\F1_AC_Digital_Twin\data\processed\MoE-anomaly
Split ratios: (0.7, 0.15, 0.15)


---

## Section 1: Feature Definitions per Expert

In [3]:
# Expert 1: Tire Dynamics (20 features)
# TIRE_FEATURES = [
#     # Temperature (4 wheels)
#     'TireTemp_FL_Avg', 'TireTemp_FR_Avg',
#     'TireTemp_RL_Avg', 'TireTemp_RR_Avg',
    
#     # Wear (4 wheels)
#     'TireWear_FL', 'TireWear_FR',
#     'TireWear_RL', 'TireWear_RR',
    
#     # Pressure (4 wheels)
#     'TirePressure_FL', 'TirePressure_FR',
#     'TirePressure_RL', 'TirePressure_RR',
    
#     # Slip Ratio (4 wheels)
#     'SlipRatio_FL', 'SlipRatio_FR',
#     'SlipRatio_RL', 'SlipRatio_RR',
    
#     # Slip Angle (4 wheels)
#     'SlipAngle_FL', 'SlipAngle_FR',
#     'SlipAngle_RL', 'SlipAngle_RR',
# ]

# Expert 2: Vehicle Dynamics (16 features) - UPDATED
DYNAMICS_FEATURES = [
    # G-forces
    'AccG_Lateral', 'AccG_Vertical', 'AccG_Longitudinal',
    
    # Local velocities
    'LocalVelocity_X', 'LocalVelocity_Y', 'LocalVelocity_Z',
    
    # Angular velocities
    'AngularVel_X', 'AngularVel_Y', 'AngularVel_Z',
    
    # Orientation
    'Heading', 'Pitch', 'Roll',
    
    # Load distribution (ALL 4 wheels)
    'TireLoad_FL', 'TireLoad_FR', 'TireLoad_RL', 'TireLoad_RR',
]

# Expert 3: Driver Control (6 features) - UPDATED
# Note: TC_InAction, ABS_InAction, BrakeBias, and BrakeTemp variables removed due to zero variance
CONTROL_FEATURES = [
    # Basic controls (only variables with variance in telemetry data)
    'Speed_kmh', 'RPM', 'Throttle', 'Brake', 'Steering', 'Gear',
]

# Expert 4: Power Systems (4 features) - UPDATED
# Note: Fuel, EngineTemp_Oil, CurrentMaxRpm, EngineBrake, ERS_PowerLevel, ERS_RecoveryLevel removed (zero variance)
POWER_FEATURES = [
    'TurboBoost',
    'KERS_Charge', 'KERS_CurrentKJ',
    'DRS_Enabled',
]

# Metadata columns for context
META_COLS = ['Timestamp', 'CompletedLaps', 'DistanceTraveled_m', 'CurrentSectorIndex', 'IsInPit']

# All expert features combined
ALL_EXPERT_FEATURES = DYNAMICS_FEATURES + CONTROL_FEATURES + POWER_FEATURES

#print(f"Expert 1 (Tire): {len(TIRE_FEATURES)} features")
print(f"Expert 2 (Dynamics): {len(DYNAMICS_FEATURES)} features")
print(f"Expert 3 (Control): {len(CONTROL_FEATURES)} features")
print(f"Expert 4 (Power): {len(POWER_FEATURES)} features")
print(f"Total: {len(ALL_EXPERT_FEATURES)} features")

Expert 2 (Dynamics): 16 features
Expert 3 (Control): 6 features
Expert 4 (Power): 4 features
Total: 26 features


---

## Section 2: Load and Explore Raw Data

In [4]:
def load_telemetry_data(file_path):
    """
    Load raw telemetry CSV file.
    """
    print(f"Loading telemetry data from {file_path.name}...")
    df = pd.read_csv(file_path)
    
    print(f"✅ Loaded successfully")
    print(f"   Shape: {df.shape}")
    print(f"   Columns: {len(df.columns)}")
    print(f"   Memory: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
    
    return df

# Load data
df_raw = load_telemetry_data(INPUT_FILE)
df_raw.head(3)

Loading telemetry data from telemetry_2025-12-08_18-05-21.csv...
✅ Loaded successfully
   Shape: (66125, 164)
   Columns: 164
   Memory: 85.5 MB


,Timestamp,Speed_kmh,RPM,Throttle,Brake,Clutch,Steering,Gear,Velocity_X,Velocity_Y,...,CarX,CarY,CarZ,FinalFF,PerformanceMeter,IsAIControlled,P2P_Activations,P2P_Status,Distance,DistanceTraveled_m_Original
0,1765206845,79.91,9921,0.62,0.0,1.0,0.009,1,1.656,0.092,...,-156.590,-6.958,104.214,0.014,0.12,True,0,0,0.000,38876.941
1,1765206845,80.20,9768,0.61,0.0,1.0,0.009,1,1.693,0.120,...,-156.420,-6.947,101.952,-0.019,0.12,True,0,0,2.243,38879.184
2,1765206845,80.25,9536,0.60,0.0,1.0,0.011,1,1.730,0.116,...,-156.256,-6.935,99.822,-0.007,0.12,True,0,0,4.407,38881.348


In [5]:
def check_feature_availability(df, feature_lists):
    """
    Check if all required features exist in dataframe.
    """
    available_cols = set(df.columns)
    missing = {}
    
    for expert_name, features in feature_lists.items():
        missing_feats = [f for f in features if f not in available_cols]
        if missing_feats:
            missing[expert_name] = missing_feats
    
    return missing

# Check feature availability
feature_lists = {
    #'Expert 1 (Tire)': TIRE_FEATURES,
    'Expert 2 (Dynamics)': DYNAMICS_FEATURES,
    'Expert 3 (Control)': CONTROL_FEATURES,
    'Expert 4 (Power)': POWER_FEATURES,
    'Metadata': META_COLS
}

missing_features = check_feature_availability(df_raw, feature_lists)

if missing_features:
    print("⚠️ Missing features:")
    for expert, feats in missing_features.items():
        print(f"\n{expert}:")
        for f in feats:
            print(f"  - {f}")
else:
    print("All required features are available")

All required features are available


---

## Section 3: Data Cleaning

In [6]:
def clean_data(df, feature_cols):
    """
    Clean data: handle missing values and infinities.
    """
    df_clean = df.copy()
    
    # Replace infinities with NaN
    df_clean[feature_cols] = df_clean[feature_cols].replace([np.inf, -np.inf], np.nan)
    
    # Fill missing values with forward fill, then backward fill
    df_clean[feature_cols] = df_clean[feature_cols].fillna(method='ffill').fillna(method='bfill')
    
    # If still NaN, fill with column mean
    df_clean[feature_cols] = df_clean[feature_cols].fillna(df_clean[feature_cols].mean())
    
    return df_clean

# Clean data
df_clean = clean_data(df_raw, ALL_EXPERT_FEATURES)
print(f"Data cleaned. Remaining NaN values: {df_clean[ALL_EXPERT_FEATURES].isna().sum().sum()}")

Data cleaned. Remaining NaN values: 0


C:\Users\victo\AppData\Local\Temp\ipykernel_29588\1614627877.py:11: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_clean[feature_cols] = df_clean[feature_cols].fillna(method='ffill').fillna(method='bfill')


---

## Section 4: Train/Val/Test Split by Laps

In [7]:
def split_by_laps(df, split_ratios=(0.7, 0.15, 0.15), seed=42):
    """
    Split data by complete laps to avoid data leakage.
    """
    # Get unique laps
    unique_laps = df['CompletedLaps'].unique()
    n_laps = len(unique_laps)
    
    # Shuffle laps
    np.random.seed(seed)
    shuffled_laps = np.random.permutation(unique_laps)
    
    # Calculate split indices
    n_train = int(n_laps * split_ratios[0])
    n_val = int(n_laps * split_ratios[1])
    
    # Split laps
    train_laps = shuffled_laps[:n_train]
    val_laps = shuffled_laps[n_train:n_train + n_val]
    test_laps = shuffled_laps[n_train + n_val:]
    
    # Create splits
    splits = {
        'train': df[df['CompletedLaps'].isin(train_laps)].copy(),
        'val': df[df['CompletedLaps'].isin(val_laps)].copy(),
        'test': df[df['CompletedLaps'].isin(test_laps)].copy()
    }
    
    # Summary
    print("=" * 60)
    print("DATA SPLIT SUMMARY")
    print("=" * 60)
    for split_name, split_df in splits.items():
        n_laps_split = split_df['CompletedLaps'].nunique()
        pct = n_laps_split / n_laps * 100
        print(f"\n{split_name.upper()}:")
        print(f"  Laps: {n_laps_split} ({pct:.1f}%)")
        print(f"  Samples: {len(split_df):,}")
    
    return splits

# Create splits
splits = split_by_laps(df_clean, SPLIT_RATIOS, SEED)

DATA SPLIT SUMMARY

TRAIN:
  Laps: 56 (70.0%)
  Samples: 46,260

VAL:
  Laps: 12 (15.0%)
  Samples: 9,781

TEST:
  Laps: 12 (15.0%)
  Samples: 10,084


---

## Section 5: Feature Extraction and Normalization per Expert

In [8]:
def prepare_expert_data(splits, feature_cols, expert_name):
    """
    Extract and normalize features for a specific expert.
    """
    print(f"\nPreparing data for {expert_name}...")
    
    # Extract features
    expert_splits = {}
    for split_name, df in splits.items():
        expert_splits[split_name] = df[feature_cols + META_COLS].copy()
    
    # Fit scaler on training data
    scaler = StandardScaler()
    scaler.fit(expert_splits['train'][feature_cols])
    
    # Transform all splits
    for split_name in expert_splits.keys():
        scaled_features = scaler.transform(expert_splits[split_name][feature_cols])
        
        scaled_df = pd.DataFrame(
            scaled_features,
            columns=feature_cols,
            index=expert_splits[split_name].index
        )
        
        for meta_col in META_COLS:
            scaled_df[meta_col] = expert_splits[split_name][meta_col].values
        
        expert_splits[split_name] = scaled_df
    
    print(f"  ✅ Processed {len(feature_cols)} features")
    
    return expert_splits, scaler

# Prepare data for all experts
expert_data = {}
scalers = {}

# expert_data['expert1_tire'], scalers['expert1_tire'] = prepare_expert_data(
 #   splits, TIRE_FEATURES, 'Expert 1 (Tire Dynamics)'
#)

expert_data['expert2_dynamics'], scalers['expert2_dynamics'] = prepare_expert_data(
    splits, DYNAMICS_FEATURES, 'Expert 2 (Vehicle Dynamics)'
)

expert_data['expert3_control'], scalers['expert3_control'] = prepare_expert_data(
    splits, CONTROL_FEATURES, 'Expert 3 (Driver Control)'
)

expert_data['expert4_power'], scalers['expert4_power'] = prepare_expert_data(
    splits, POWER_FEATURES, 'Expert 4 (Power Systems)'
)


Preparing data for Expert 2 (Vehicle Dynamics)...
  ✅ Processed 16 features

Preparing data for Expert 3 (Driver Control)...
  ✅ Processed 6 features

Preparing data for Expert 4 (Power Systems)...
  ✅ Processed 4 features


---

## Section 6: Save Processed Data

In [9]:
# Save expert splits
print("\nSaving processed data...\n")

for expert_name, expert_splits in expert_data.items():
    print(f"{expert_name}:")
    for split_name, df in expert_splits.items():
        filename = f"{expert_name}_{split_name}.csv"
        filepath = SPLITS_DIR / filename
        df.to_csv(filepath, index=False)
        print(f"  Saved {filename}")

# Save scalers
print("\nSaving scalers...\n")

for expert_name, scaler in scalers.items():
    filename = f"scaler_{expert_name}.pkl"
    filepath = SCALERS_DIR / filename
    
    with open(filepath, 'wb') as f:
        pickle.dump(scaler, f)
    
    print(f"  Saved {filename}")

print("\n All data saved successfully!")


Saving processed data...

expert2_dynamics:
  Saved expert2_dynamics_train.csv
  Saved expert2_dynamics_val.csv
  Saved expert2_dynamics_test.csv
expert3_control:
  Saved expert3_control_train.csv
  Saved expert3_control_val.csv
  Saved expert3_control_test.csv
expert4_power:
  Saved expert4_power_train.csv
  Saved expert4_power_val.csv
  Saved expert4_power_test.csv

Saving scalers...

  Saved scaler_expert2_dynamics.pkl
  Saved scaler_expert3_control.pkl
  Saved scaler_expert4_power.pkl

 All data saved successfully!


---
## Summary

Data preparation complete! Ready for GMM training in the next notebooks:
- N01_MoE_expert1_tire.ipynb
- N02_MoE_expert2_dynamics.ipynb
- N03_MoE_expert3_control.ipynb
- N04_MoE_expert4_power.ipynb